# Демо проектных упражнений модуля 5 — Память внимания и цена отбора позиций

**Демо-реализация, не готовое решение полномасштабного проекта.** Два опыта: аналитическая оценка массивов и двухэтапный поиск на синтетических данных. Второй опыт иллюстрирует отбор; обучаемого индексатора DSA здесь нет.
Все входы малы, детерминированы и работают без сети на CPU. Цель — проверить
интерфейсы, метрики, инварианты и структуру отчёта до расходования GPU/API-бюджета.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"воспроизводимый тестовый запуск; seed={SEED}")
DEMO_LANGUAGE = 'ru'


## Демо 1. Что именно хранит вычисление

Сравним объём выбранных массивов при росте длины. Прямой вариант хранит выход, логиты и вероятности всех пар; потоковый — выход и перечисленные буферы одного тайла. В формулах Q/K/V, обратный проход и служебные расходы пока отсутствуют. Найдите квадратичное слагаемое до запуска графика и предскажите, какое из двух выражений будет расти быстрее.

In [ ]:
lengths = np.array([1024, 2048, 4096, 8192, 16384])
heads, width, tile, bytes_per = 16, 64, 128, 4
output = heads * lengths * width * bytes_per
dense = output + 2 * heads * lengths**2 * bytes_per
stream = output + heads * (tile**2 + tile * width + 2 * tile) * bytes_per
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 12, "axes.spines.top": False, "axes.spines.right": False})
fig, ax = plt.subplots(figsize=(11, 5))
ax.loglog(lengths, dense / 2**20, "o-", color="#5E81AC", label="output + full scores/probabilities")
ax.loglog(lengths, stream / 2**20, "o-", color="#A3BE8C", label="output + selected tile buffers")
ax.set(xlabel="sequence length", ylabel="analytical MiB", title="Selected arrays, not measured GPU peak memory")
ax.legend(); ax.grid(alpha=.25); plt.show()
assert np.all(dense > stream)
print("Excludes Q/K/V, allocator overhead, backward, and external masks. Tile buffers depend on kernel scheduling.")

При фиксированном размере тайла его буферы не растут с длиной, но массив выхода растёт линейно. В прямой оценке доминируют две квадратные матрицы. Это сравнение выбранных массивов, а не измеренный пик GPU: другой порядок вычисления или дополнительные буферы изменят полный расход.

## Демо 2. Цена короткого списка кандидатов

Запрос построен как зашумлённая копия одного из ключей, поэтому нужный индекс известен. Сначала используем только восемь координат для дешёвого отбора 64 кандидатов, затем сравниваем их в полном пространстве ширины 32. Измеряем попадание нужного ключа в список, итоговый выбор и отличие выхода внимания от полного расчёта. Последние два результата могут расходиться: правильный лучший ключ ещё не означает сохранения всей взвешенной смеси.

In [ ]:
rng = np.random.default_rng(SEED)
trials, length, dim, index_rank, topk = 200, 2048, 32, 8, 64
recall = hits = full_hits = 0
relative_errors, retained_mass = [], []
def softmax_row(x):
    e = np.exp(x - x.max())
    return e / e.sum()
for _ in range(trials):
    keys = rng.normal(size=(length, dim))
    values = rng.normal(size=(length, 6))
    target = rng.integers(length)
    query = keys[target] + 0.05 * rng.normal(size=dim)
    index_scores = keys[:, :index_rank] @ query[:index_rank]
    shortlist = np.argpartition(index_scores, -topk)[-topk:]
    scores = keys @ query / np.sqrt(dim)
    recall += int(target in shortlist)
    hits += int(shortlist[np.argmax(scores[shortlist])] == target)
    full_hits += int(np.argmax(scores) == target)
    p_full, p_short = softmax_row(scores), softmax_row(scores[shortlist])
    out_full, out_short = p_full @ values, p_short @ values[shortlist]
    retained_mass.append(p_full[shortlist].sum())
    relative_errors.append(np.linalg.norm(out_short-out_full) / max(np.linalg.norm(out_full), 1e-12))
score_ratio = (length * index_rank + topk * dim) / (length * dim)
assert score_ratio == .28125
assert 0 <= hits <= recall <= trials
print(f"shortlist recall={recall/trials:.3f}; final retrieval={hits/trials:.3f}; full retrieval={full_hits/trials:.3f}")
print(f"mean retained probability={np.mean(retained_mass):.3f}; mean relative output error={np.mean(relative_errors):.3f}")
print(f"score multiplications ratio={score_ratio:.5f}; final scoring alone={topk/length:.5f}")
print("The ratio excludes top-k, learned projections, softmax, value aggregation and data movement; it is not a speedup.")

Вместо одного коэффициента смотрим на весь набор метрик. Работа двух сопоставлений составляет 8/32 + 64/2048 = 0.28125 от полного скалярного сопоставления; сами выбранные позиции дали бы лишь 0.03125. Top-k, проекции, смешивание значений и обмен с памятью сюда ещё не входят. Уменьшите число кандидатов и проверьте, как меняются полнота списка и ошибка выхода; направление изменения на конкретной случайной выборке следует измерить.

## Переход к полному проекту

Для GPU измерьте одинаковые входы и маски после прогрева, с синхронизацией и сбросом пикового счётчика. Отдельно укажите область замера: проекции, отбор, основное внимание и кэш. В опыте с отбором качество поиска и ошибка выхода внимания — разные метрики.

### Минимальный отчёт

1. Версии данных, модели, кода и seed.
2. Основная метрика, а также качество, стоимость, задержка и память там, где это применимо.
3. Базовая линия, одинаковый бюджет и хотя бы один отрицательный контроль.
4. Неудачные прогоны и границы обоснованных выводов.